In [ ]:
# =========================
# 02 MODEL TRAINING
# Serie A Match Prediction
# =========================

import pandas as pd
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

import joblib

# =========================
# LOAD PROCESSED DATASET
# =========================

df_final = pd.read_csv("df_final.csv")

df_final.head()

In [ ]:
# =========================
# BASIC CHECKS
# =========================

print(df_final.shape)
print(df_final.isna().sum())
print(df_final["FTR"].value_counts())

In [ ]:
# =========================
# CREATE X AND y
# =========================

X = df_final.drop(
    columns=["Date", "HomeTeam", "AwayTeam", "FTR"]
)

y = df_final["FTR"]

print(X.head())
print(y.head())

In [ ]:
# =========================
# TEMPORAL TRAIN / TEST SPLIT
# =========================

split = int(len(df_final) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print("Train size:", X_train.shape)
print("Test size:", X_test.shape)

In [ ]:
# =========================
# TRAIN LOGISTIC REGRESSION MODEL
# =========================

model = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

model.fit(X_train, y_train)

In [ ]:
# =========================
# MAKE PREDICTIONS
# =========================

predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

In [ ]:
# =========================
# CLASSIFICATION REPORT
# =========================

print(classification_report(y_test, predictions))

In [ ]:
# =========================
# CONFUSION MATRIX
# =========================

cm = confusion_matrix(y_test, predictions, labels=model.classes_)

cm_df = pd.DataFrame(
    cm,
    index=[f"Real_{c}" for c in model.classes_],
    columns=[f"Pred_{c}" for c in model.classes_]
)

cm_df

In [ ]:
# =========================
# SAVE PREDICTIONS
# =========================

df_predictions = df_final.iloc[split:].copy()

df_predictions["Predicted_Result"] = predictions

probabilities = model.predict_proba(X_test)

prob_df = pd.DataFrame(
    probabilities,
    columns=[f"Prob_{c}" for c in model.classes_]
)

df_predictions = pd.concat(
    [df_predictions.reset_index(drop=True), prob_df.reset_index(drop=True)],
    axis=1
)

df_predictions.head()

In [ ]:
# =========================
# SAVE MODEL AND OUTPUTS
# =========================

joblib.dump(model, "logistic_model.pkl")

df_predictions.to_csv("predictions.csv", index=False)

cm_df.to_csv("confusion_matrix.csv")

print("Model and outputs saved successfully.")

In [ ]:
# =========================
# FEATURE COEFFICIENTS
# =========================

coef_df = pd.DataFrame(
    model.coef_,
    columns=X.columns,
    index=model.classes_
)

coef_df